In [0]:
%pip install --upgrade databricks-sdk

In [0]:
dbutils.library.restartPython()

In [0]:
import os

policy_json_path = os.path.abspath("policy.json")
print(policy_json_path)

In [0]:
import json

def read_policy(polity_path: str) -> dict:
    with open(polity_path, 'r') as f:
        policy_dict = json.load(f)

    return policy_dict

In [0]:
import pandas as pd
from databricks.sdk import WorkspaceClient

class DatabricksEndpointAdapter:
    """
    Adapter to list model serving endpoints in Databricks and return them as a pandas DataFrame.
    """
    def __init__(self, workspace_client: WorkspaceClient = None):
        self.client = workspace_client or WorkspaceClient()

    def list_serving_endpoints(self):
        endpoints = self.client.serving_endpoints.list()
        # Convert SDK objects to dictionaries to handle complex nested types
        endpoint_dicts = [ep.as_dict() for ep in endpoints]
        return pd.DataFrame(endpoint_dicts)
    
    def get_serving_endpoint_details(self, endpoint_id: str) -> dict:
        endpoint = self.client.serving_endpoints.get(endpoint_id)
        return endpoint.as_dict()

In [0]:
my_endpoints_df = DatabricksEndpointAdapter().list_serving_endpoints()
my_endpoints_df.head(5)

In [0]:
import re

def apply_to_filter(endpoints_df: pd.DataFrame, apply_to_filter: dict)-> pd.DataFrame:
    """
    Filter the pandas using a dictionary of columns and regular expresion to match
    example: {"name": "^databricks-.*"} filter column name seaching for regex match of ^databricks-.*
    """

    mask = pd.Series([True] * len(endpoints_df))
    for col, pattern in apply_to_filter.items():
        mask &= endpoints_df[col].astype(str).str.match(pattern)
    return endpoints_df[mask]
    

In [0]:
filter = {"name": "^datab*"}
applied_endpoints_df = apply_to_filter(my_endpoints_df, filter)
display(applied_endpoints_df)

In [0]:
dea = DatabricksEndpointAdapter()
first_policy = read_policy("policy.json")
   

In [0]:
dea = DatabricksEndpointAdapter()

detail_json = dea.get_serving_endpoint_details("databricks-claude-opus-4-5")
detail_json

In [0]:
policy_rules = first_policy.get("rules")
policy_rules

In [0]:
def seach_policy_key(d: dict, key: str) -> tuple[bool, object]:
    """
    Check if a (possibly nested) key exists in a dictionary using dot notation and return its value if found.

    Args:
        d (dict): The dictionary to search.
        key (str): The key to search for, using dot notation for nested keys (e.g., "config.rate_limit").

    Returns:
        (bool, value): Tuple of (True, value) if the key exists, (False, None) otherwise.

    Examples:
        >>> d = {"config": {"rate_limit": 100}}
        >>> seach_policy_key(d, "config.rate_limit")
        (True, 100)

        >>> d = {"config": {"rate_limit": 100}}
        >>> seach_policy_key(d, "config.timeout")
        (False, None)

        >>> d = {"name": "endpoint"}
        >>> seach_policy_key(d, "name")
        (True, "endpoint")
    """
    keys = key.split(".")
    current = d
    for k in keys[:-1]:
        if isinstance(current, dict) and k in current:
            current = current[k]
        else:
            return False, None
    if isinstance(current, dict) and keys[-1] in current:
        return True, current[keys[-1]]
    return False, None

# Testing seach_policy_key
test_cases = [
    ({"config": {"rate_limit": 100}}, "config.rate_limit", (True, 100)),
    ({"config": {"rate_limit": 100}}, "config.timeout", (False, None)),
    ({"name": "endpoint"}, "name", (True, "endpoint")),
    ({"a": {"b": {"c": 1}}}, "a.b.c", (True, 1)),
    ({"a": {"b": {"c": 1}}}, "a.b.d", (False, None)),
    ({}, "missing", (False, None)),
]

for i, (d, key, expected) in enumerate(test_cases):
    result = seach_policy_key(d, key)
    print(f"Test case {i} was {'✔️' if result == expected else '❌'}: seach_policy_key({d}, {key}) = {result}, expected {expected}")

In [0]:
def apply_policy_v2(policy_rules: dict, detail_json: dict):
    """
    Check if detail_json satisfies all rules in policy_rules.
    Returns (True, detail_json, errors) if all rules are satisfied.
    If not, returns (False, new_detail_json, errors) where new_detail_json is detail_json updated to fit the rules.
    """
    errors = []
    new_detail_json = detail_json.copy()
    for key, rule in policy_rules.items():
        rule_type = rule.get("type")
        default = rule.get("default")
        exists, value = seach_policy_key(detail_json, key)
        if rule_type == "required":
            if not exists:
                errors.append({"key": key, "error": f"Missing required key: {key}"})
        elif rule_type == "fixed":            
            if not exists or value != default:
                # Set the value in the top-level dict only if not nested
                if "." not in key:
                    new_detail_json[key] = default
                errors.append({"key": key, "error": f"Key {key} is not fixed to required value"})
    if not errors:
        return True, detail_json, errors
    else:
        return False, new_detail_json, errors

result, checked_json, errors = apply_policy_v2(policy_rules, detail_json)
print("result,:  ", result)
print("checked_json:",  checked_json)
print("Errors:", errors)

In [0]:
def normalize_rate_limits(endpoint_json: dict) -> dict:
    """
    Transform rate_limits from array format to dictionary format for easier policy validation.
    
    Input:  {"ai_gateway": {"rate_limits": [{"key": "user", "calls": 0, "renewal_period": "minute"}]}}
    Output: {"ai_gateway": {"rate_limits": {"user": {"requests_per_minute": 0}}}}
    """
    normalized = endpoint_json.copy()
    
    if "ai_gateway" in normalized and "rate_limits" in normalized["ai_gateway"]:
        rate_limits_array = normalized["ai_gateway"]["rate_limits"]
        
        if isinstance(rate_limits_array, list):
            # Convert array to dictionary
            rate_limits_dict = {}
            for limit in rate_limits_array:
                key = limit.get("key")
                calls = limit.get("calls", 0)
                renewal_period = limit.get("renewal_period", "minute")
                
                # Map to requests_per_minute/hour/day based on renewal_period
                field_name = f"requests_per_{renewal_period}"
                rate_limits_dict[key] = {field_name: calls}
            
            normalized["ai_gateway"]["rate_limits"] = rate_limits_dict
    
    return normalized

# Test the normalization
test_endpoint = {
    "ai_gateway": {
        "rate_limits": [
            {"calls": 0, "key": "user", "renewal_period": "minute"},
            {"calls": 100, "key": "AZ_DATAANALYTICSRDPRE_LKHSRD_Oper", "renewal_period": "minute"}
        ]
    }
}

normalized = normalize_rate_limits(test_endpoint)
print("Original:")
print(test_endpoint)
print("\nNormalized:")
print(normalized)

In [0]:
def apply_policy_v3(policy_rules: dict, detail_json: dict, normalize: bool = True):
    """
    Check if detail_json satisfies all rules in policy_rules.
    
    Args:
        policy_rules: Dictionary of policy rules
        detail_json: Endpoint configuration to validate
        normalize: If True, normalize rate_limits array to dict format before validation
    
    Returns:
        (result, checked_json, errors): 
            - result: True if all rules satisfied
            - checked_json: Original or normalized JSON
            - errors: List of validation errors
    """
    errors = []
    
    # Normalize if requested
    working_json = normalize_rate_limits(detail_json) if normalize else detail_json.copy()
    
    for key, rule in policy_rules.items():
        rule_type = rule.get("type")
        default = rule.get("default")
        error_message = rule.get("error_message", f"Policy violation for {key}")
        
        exists, value = seach_policy_key(working_json, key)
        
        if rule_type == "required":
            if not exists:
                errors.append({"key": key, "error": error_message})
        
        elif rule_type == "fixed":
            if not exists:
                errors.append({"key": key, "error": f"{error_message} (key missing)"})
            elif value != default:
                errors.append({
                    "key": key, 
                    "error": f"{error_message} (expected: {default}, found: {value})"
                })
    
    result = len(errors) == 0
    return result, working_json, errors

# Test with the new policy
test_policy = {
    "ai_gateway.rate_limits": {
        "type": "required",
        "error_message": "El AI Gateway debe estar presente aunque el endpoint esté deshabilitado"
    },
    "ai_gateway.rate_limits.user.requests_per_minute": {
        "type": "fixed",
        "default": 0,
        "error_message": "El rate limit de user está fijado a 0 (endpoint deshabilitado)"
    },
    "ai_gateway.rate_limits.AZ_DATAANALYTICSRDPRE_LKHSRD_Oper.requests_per_minute": {
        "type": "fixed",
        "default": 0,
        "error_message": "El rate limit de grupo está fijado a 0 (endpoint deshabilitado)"
    }
}

result, checked_json, errors = apply_policy_v3(test_policy, test_endpoint)
print("\n=== VALIDATION RESULTS ===")
print(f"Result: {result}")
print(f"\nErrors: {errors}")
print(f"\nNormalized JSON:")
print(checked_json)

In [0]:
# Test with the real endpoint from cell 12
real_endpoint = detail_json

# Define a comprehensive policy
comprehensive_policy = {
    "ai_gateway.rate_limits": {
        "type": "required",
        "error_message": "El AI Gateway debe estar presente aunque el endpoint esté deshabilitado"
    },
    "ai_gateway.rate_limits.user.requests_per_minute": {
        "type": "fixed",
        "default": 0,
        "error_message": "El rate limit de user está fijado a 0 (endpoint deshabilitado)"
    },"ai_gateway.rate_limits.AZ_DATAANALYTICSRDPRE_LKHSRD_Oper.requests_per_minute": {
        "type": "fixed",
        "default": 0,
        "error_message": "El rate limit de grupo está fijado a 0 (endpoint deshabilitado)"
    }
}

result, normalized_endpoint, errors = apply_policy_v3(comprehensive_policy, real_endpoint)

print("=== VALIDACIÓN DEL ENDPOINT REAL ===")
print(f"Endpoint: {real_endpoint.get('name')}")
print(f"\nCumple política: {result}")
print(f"\nErrores encontrados: {len(errors)}")
for error in errors:
    print(f"  - {error['key']}: {error['error']}")

print(f"normalized_endpoint : {normalized_endpoint}")

In [0]:
def parse_rate_limit_key(policy_key: str) -> dict:
    """
    Parse a policy key like 'ai_gateway.rate_limits.user.requests_per_minute'
    into components for array manipulation.
    
    Returns:
        {
            'is_rate_limit': bool,
            'key_name': str (e.g., 'user'),
            'field': str (e.g., 'requests_per_minute'),
            'renewal_period': str (e.g., 'minute')
        }
    """
    parts = policy_key.split('.')
    
    # Check if it's a rate_limits policy
    if len(parts) >= 4 and parts[0] == 'ai_gateway' and parts[1] == 'rate_limits':
        key_name = parts[2]  # e.g., 'user', 'AZ_DATAANALYTICSRDPRE_LKHSRD_Oper'
        field = parts[3]     # e.g., 'requests_per_minute'
        
        # Extract renewal_period from field name
        renewal_period = field.replace('requests_per_', '') if field.startswith('requests_per_') else 'minute'
        
        return {
            'is_rate_limit': True,
            'key_name': key_name,
            'field': field,
            'renewal_period': renewal_period
        }
    
    return {'is_rate_limit': False}

# Test parsing
test_keys = [
    'ai_gateway.rate_limits.user.requests_per_minute',
    'ai_gateway.rate_limits.AZ_DATAANALYTICSRDPRE_LKHSRD_Oper.requests_per_minute',
    'ai_gateway.rate_limits',
    'config.served_entities'
]

for key in test_keys:
    parsed = parse_rate_limit_key(key)
    print(f"{key}")
    print(f"  → {parsed}\n")

In [0]:
def apply_policy_v4(policy_rules: dict, detail_json: dict):
    """
    Validate and apply fixes to endpoint configuration according to policy rules.
    Preserves the original endpoint structure (arrays, nested objects).
    Always applies corrections - use dry_mode at module level to control actual updates.
    
    Args:
        policy_rules: Dictionary of policy rules
        detail_json: Endpoint configuration to validate
    
    Returns:
        (result, corrected_json, errors):
            - result: True if all rules satisfied (after applying fixes)
            - corrected_json: Endpoint JSON with fixes applied
            - errors: List of validation errors found before fixes
    """
    errors = []
    corrected_json = detail_json.copy()
    
    for policy_key, rule in policy_rules.items():
        rule_type = rule.get("type")
        default = rule.get("default")
        error_message = rule.get("error_message", f"Policy violation for {policy_key}")
        
        # Parse the policy key to understand its structure
        parsed = parse_rate_limit_key(policy_key)
        
        if parsed['is_rate_limit']:
            # Handle rate_limits array structure
            key_name = parsed['key_name']
            renewal_period = parsed['renewal_period']
            
            # Navigate to rate_limits array
            if 'ai_gateway' not in corrected_json:
                if rule_type == 'required':
                    errors.append({"key": policy_key, "error": f"{error_message} (ai_gateway missing)"})
                continue
                
            if 'rate_limits' not in corrected_json['ai_gateway']:
                if rule_type == 'required':
                    errors.append({"key": policy_key, "error": f"{error_message} (rate_limits missing)"})
                    if rule_type == 'fixed':
                        corrected_json['ai_gateway']['rate_limits'] = []
                continue
            
            rate_limits = corrected_json['ai_gateway']['rate_limits']
            
            # Check if it's just checking for rate_limits existence
            if policy_key == 'ai_gateway.rate_limits':
                if rule_type == 'required' and not rate_limits:
                    errors.append({"key": policy_key, "error": error_message})
                continue
            
            # Find the specific rate limit entry by key
            limit_entry = None
            limit_index = None
            for idx, limit in enumerate(rate_limits):
                if limit.get('key') == key_name:
                    limit_entry = limit
                    limit_index = idx
                    break
            
            if rule_type == 'required':
                if limit_entry is None:
                    errors.append({"key": policy_key, "error": f"{error_message} (key '{key_name}' not found)"})
            
            elif rule_type == 'fixed':
                if limit_entry is None:
                    errors.append({"key": policy_key, "error": f"{error_message} (key '{key_name}' missing)"})
                    # Always add new rate limit entry
                    corrected_json['ai_gateway']['rate_limits'].append({
                        'key': key_name,
                        'calls': default,
                        'renewal_period': renewal_period
                    })
                else:
                    current_value = limit_entry.get('calls')
                    if current_value != default:
                        errors.append({
                            "key": policy_key,
                            "error": f"{error_message} (expected: {default}, found: {current_value})"
                        })
                        # Always update the calls value in the array
                        corrected_json['ai_gateway']['rate_limits'][limit_index]['calls'] = default
        
        else:
            # Handle non-rate_limit keys using the existing seach_policy_key function
            exists, value = seach_policy_key(corrected_json, policy_key)
            
            if rule_type == 'required':
                if not exists:
                    errors.append({"key": policy_key, "error": error_message})
            
            elif rule_type == 'fixed':
                if not exists:
                    errors.append({"key": policy_key, "error": f"{error_message} (key missing)"})
                elif value != default:
                    errors.append({
                        "key": policy_key,
                        "error": f"{error_message} (expected: {default}, found: {value})"
                    })
    
    result = len(errors) == 0
    return result, corrected_json, errors

# Create a fresh test endpoint with ARRAY structure (not normalized)
test_endpoint_array = {
    "ai_gateway": {
        "rate_limits": [
            {"calls": 0, "key": "user", "renewal_period": "minute"},
            {"calls": 100, "key": "AZ_DATAANALYTICSRDPRE_LKHSRD_Oper", "renewal_period": "minute"}
        ]
    }
}

# Test validation and automatic fixes
print("=== TEST: Validation with automatic fixes ===")
result, corrected, errors = apply_policy_v4(test_policy, test_endpoint_array)
print(f"Errors found: {errors}")
print(f"Result after fixes: {result}")
print(f"\nOriginal endpoint:")
print(test_endpoint_array.get('ai_gateway', {}).get('rate_limits'))
print(f"\nCorrected endpoint (array structure maintained):")
print(corrected.get('ai_gateway', {}).get('rate_limits'))
print(f"\n💡 Use dry_mode at module level to control if corrected_json is applied to the endpoint")

In [0]:
# Fetch the real endpoint fresh to ensure original structure
real_endpoint = dea.get_serving_endpoint_details("databricks-claude-opus-4-5")

# Define a comprehensive policy
comprehensive_policy = {
    "ai_gateway.rate_limits": {
        "type": "required",
        "error_message": "El AI Gateway debe estar presente aunque el endpoint esté deshabilitado"
    },
    "ai_gateway.rate_limits.user.requests_per_minute": {
        "type": "fixed",
        "default": 0,
        "error_message": "El rate limit de user está fijado a 0 (endpoint deshabilitado)"
    },
    "ai_gateway.rate_limits.AZ_DATAANALYTICSRDPRE_LKHSRD_Oper.requests_per_minute": {
        "type": "fixed",
        "default": 100,
        "error_message": "El rate limit de grupo está fijado a 100 requests/minute"
    }
}

print("=== VALIDACIÓN DEL ENDPOINT REAL ===")
print(f"Endpoint: {real_endpoint.get('name')}")
print(f"\nRate limits originales:")
print(real_endpoint.get('ai_gateway', {}).get('rate_limits'))

# Validate and apply fixes (always)
result, corrected_endpoint, errors = apply_policy_v4(comprehensive_policy, real_endpoint)
print(f"\n--- Validación y corrección ---")
print(f"Errores encontrados: {errors}")
print(f"Cumple política después de fixes: {result}")
print(f"\nRate limits corregidos:")
print(corrected_endpoint.get('ai_gateway', {}).get('rate_limits'))

# Verify structure is compatible with ServingEndpointDetailed
print(f"\n--- Verificación de estructura ---")
print(f"Tipo de rate_limits: {type(corrected_endpoint.get('ai_gateway', {}).get('rate_limits'))}")
print(f"Estructura preservada para from_dict(): ✅")
print(f"\n💡 En tu módulo, usa dry_mode para decidir si aplicar corrected_endpoint al endpoint real")

In [0]:
# Test case: endpoint missing a required rate limit
endpoint_missing_limit = {
    "name": "test-endpoint",
    "ai_gateway": {
        "rate_limits": [
            {"calls": 50, "key": "user", "renewal_period": "minute"}
        ]
    }
}

policy_with_new_limit = {
    "ai_gateway.rate_limits.user.requests_per_minute": {
        "type": "fixed",
        "default": 50,
        "error_message": "User rate limit debe ser 50"
    },
    "ai_gateway.rate_limits.admin_group.requests_per_minute": {
        "type": "fixed",
        "default": 200,
        "error_message": "Admin group rate limit debe ser 200"
    }
}

print("=== TEST: Añadir rate limit faltante ===")
print(f"\nRate limits originales:")
print(endpoint_missing_limit['ai_gateway']['rate_limits'])

result, fixed_endpoint, errors = apply_policy_v4(policy_with_new_limit, endpoint_missing_limit)

print(f"\nErrores detectados: {errors}")
print(f"\nRate limits después de aplicar policy:")
print(fixed_endpoint['ai_gateway']['rate_limits'])
print(f"\n✅ Se añadió el rate limit faltante para 'admin_group'")

## ✅ Solución Completa: Normalizar Política → Endpoint

### Ventajas de este enfoque:

1. **Preserva la estructura original del endpoint** (arrays, objetos anidados)
2. **Compatible con `ServingEndpointDetailed.from_dict()`** - puedes usar el JSON corregido directamente
3. **Aplica correcciones automáticas** con `apply_fixes=True`:
   - Modifica valores existentes en el array
   - Añade entradas faltantes al array
4. **Validación clara** - reporta errores específicos con mensajes personalizados

### Flujo de trabajo:

```python
# 1. Obtener configuración actual del endpoint
endpoint_json = dea.get_serving_endpoint_details("my-endpoint")

# 2. Definir política
policy = {
    "ai_gateway.rate_limits.user.requests_per_minute": {
        "type": "fixed",
        "default": 0,
        "error_message": "User rate limit debe ser 0"
    }
}

# 3. Validar y aplicar correcciones
result, corrected_json, errors = apply_policy_v4(policy, endpoint_json, apply_fixes=True)

# 4. Usar el JSON corregido con el SDK
if not result and apply_fixes:
    # El JSON corregido mantiene la estructura original
    # Puedes usarlo con ServingEndpointDetailed.from_dict(corrected_json)
    # y luego actualizar el endpoint con la API
    pass
```

In [0]:
from databricks.sdk.service.serving import ServingEndpointDetailed

# Get current endpoint
endpoint_name = "databricks-claude-opus-4-5"
current_config = dea.get_serving_endpoint_details(endpoint_name)

# Define policy to add a new rate limit
policy_add_limit = {
    "ai_gateway.rate_limits.data_team.requests_per_minute": {
        "type": "fixed",
        "default": 50,
        "error_message": "Data team debe tener rate limit de 50 req/min"
    }
}

# Apply policy (always applies fixes)
result, corrected_config, errors = apply_policy_v4(policy_add_limit, current_config)

print("=== EJEMPLO: Actualizar endpoint con configuración corregida ===")
print(f"\nEndpoint: {endpoint_name}")
print(f"Cumple política: {result}")
print(f"\nErrores detectados: {len(errors)}")
for error in errors:
    print(f"  - {error['error']}")

print(f"\nRate limits corregidos:")
for limit in corrected_config.get('ai_gateway', {}).get('rate_limits', []):
    print(f"  - {limit['key']}: {limit['calls']} calls/{limit['renewal_period']}")

print(f"\n--- Siguiente paso ---")
print(f"El JSON corregido puede usarse con:")
print(f"  ServingEndpointDetailed.from_dict(corrected_config)")
print(f"\nY luego actualizar el endpoint con la API de Databricks.")
print(f"\n💡 En tu módulo superior, usa dry_mode para decidir si aplicar los cambios")
print(f"   - dry_mode=True: Solo reporta errores y cambios propuestos")
print(f"   - dry_mode=False: Aplica corrected_config al endpoint real")

## 🚀 Plan de Productivización: AI Gateway Policy Manager

### Estructura del Paquete Python

```
ai_gateway_policy_manager/
├── pyproject.toml                 # Configuración del paquete (Poetry/setuptools)
├── README.md
├── src/
│   └── ai_gateway_policy_manager/
│       ├── __init__.py
│       ├── adapters/
│       │   ├── __init__.py
│       │   └── databricks_adapter.py    # DatabricksEndpointAdapter
│       ├── core/
│       │   ├── __init__.py
│       │   ├── policy_engine.py         # apply_policy_v4, parse_rate_limit_key
│       │   ├── policy_loader.py         # read_policy
│       │   └── filters.py               # apply_to_filter
│       ├── models/
│       │   ├── __init__.py
│       │   ├── policy.py                # Policy, PolicyRule (dataclasses)
│       │   └── results.py               # PolicyResult, ValidationError
│       └── cli/
│           ├── __init__.py
│           └── main.py                  # CLI commands (opcional)
├── tests/
│   ├── __init__.py
│   ├── test_policy_engine.py
│   ├── test_adapters.py
│   └── fixtures/
│       └── sample_policies.json
└── examples/
    ├── basic_usage.py
    └── notebooks/
        └── example_notebook.py
```

### API Propuesta (Uso desde Notebook)

```python
# Instalación
%pip install ai-gateway-policy-manager

# Uso simple
from ai_gateway_policy_manager import PolicyManager

# Inicializar
manager = PolicyManager()

# Cargar política desde archivo
policy = manager.load_policy("policy.json")

# Aplicar a un endpoint específico
result = manager.apply_policy(
    endpoint_name="databricks-claude-opus-4-5",
    policy=policy,
    dry_mode=True  # Solo validar, no aplicar
)

# Ver resultados
print(f"Cumple política: {result.is_compliant}")
for error in result.errors:
    print(f"  - {error.message}")

# Aplicar a múltiples endpoints con filtro
results = manager.apply_policy_bulk(
    filter={"name": "^databricks-.*"},
    policy=policy,
    dry_mode=False  # Aplicar cambios
)
```

## 📋 Next Actions (Orden de Prioridad)

### 🔴 Fase 1: Core Functionality (1-2 semanas)

1. **Crear estructura del paquete**
   - [ ] Inicializar proyecto con Poetry o setuptools
   - [ ] Configurar `pyproject.toml` con dependencias
   - [ ] Crear estructura de carpetas

2. **Refactorizar código existente en módulos**
   - [ ] Mover `apply_policy_v4` → `core/policy_engine.py`
   - [ ] Mover `DatabricksEndpointAdapter` → `adapters/databricks_adapter.py`
   - [ ] Mover `read_policy`, `apply_to_filter` → módulos correspondientes
   - [ ] Añadir type hints completos
   - [ ] Añadir docstrings (Google/NumPy style)

3. **Crear modelos de datos (dataclasses/Pydantic)**
   ```python
   @dataclass
   class PolicyRule:
       type: Literal["required", "fixed"]
       default: Optional[Any]
       error_message: str
   
   @dataclass
   class PolicyResult:
       is_compliant: bool
       errors: List[ValidationError]
       corrected_config: dict
       endpoint_name: str
   ```

4. **Implementar PolicyManager (clase principal)**
   - [ ] Método `load_policy(path)` o `load_policy(dict)`
   - [ ] Método `apply_policy(endpoint_name, policy, dry_mode)`
   - [ ] Método `apply_policy_bulk(filter, policy, dry_mode)`
   - [ ] Manejo de errores robusto

### 🟡 Fase 2: Testing & Quality (1 semana)

5. **Tests unitarios**
   - [ ] Tests para `policy_engine.py` (casos edge)
   - [ ] Tests para `parse_rate_limit_key`
   - [ ] Tests para `apply_to_filter`
   - [ ] Mocks para Databricks SDK
   - [ ] Coverage > 80%

6. **Validación de políticas**
   - [ ] JSON Schema para validar estructura de políticas
   - [ ] Validación de tipos de reglas
   - [ ] Mensajes de error claros

### 🟢 Fase 3: Packaging & Distribution (3-5 días)

7. **Build & Distribution**
   - [ ] Configurar build con Poetry/setuptools
   - [ ] Generar wheel: `poetry build` o `python -m build`
   - [ ] Probar instalación local: `pip install dist/*.whl`
   - [ ] Documentación de instalación

8. **Distribución interna**
   - [ ] Subir a repositorio privado (Azure Artifacts, JFrog, etc.)
   - [ ] O: Subir a DBFS/Volumes para instalación directa
   - [ ] Documentar proceso de instalación en notebooks

### 🔵 Fase 4: Features Avanzados (Opcional)

9. **CLI (Command Line Interface)**
   ```bash
   ai-gateway-policy validate --policy policy.json --endpoint my-endpoint
   ai-gateway-policy apply --policy policy.json --filter "^databricks-.*" --dry-mode
   ```

10. **Logging & Observability**
    - [ ] Logging estructurado
    - [ ] Métricas de compliance
    - [ ] Reportes en formato JSON/CSV

11. **Extensibilidad**
    - [ ] Plugin system para custom validators
    - [ ] Soporte para otros tipos de políticas (no solo rate_limits)
    - [ ] Webhooks para notificaciones

In [0]:
# ============================================
# EJEMPLO: Cómo se usaría el paquete final
# ============================================

# Este código NO funciona aún - es la visión del API final

"""
from ai_gateway_policy_manager import PolicyManager, Policy

# Opción 1: Uso simple
manager = PolicyManager()
policy = manager.load_policy("policy.json")

result = manager.apply_policy(
    endpoint_name="databricks-claude-opus-4-5",
    policy=policy,
    dry_mode=True
)

print(f"Endpoint: {result.endpoint_name}")
print(f"Compliant: {result.is_compliant}")
print(f"Errors: {len(result.errors)}")

if not result.is_compliant:
    print("\nChanges to apply:")
    for error in result.errors:
        print(f"  - {error.key}: {error.message}")
    
    # Aplicar cambios
    if input("Apply changes? (y/n): ") == "y":
        manager.apply_policy(
            endpoint_name="databricks-claude-opus-4-5",
            policy=policy,
            dry_mode=False
        )

# Opción 2: Bulk application
results = manager.apply_policy_bulk(
    filter={"name": "^databricks-.*"},
    policy=policy,
    dry_mode=True
)

print(f"\nProcessed {len(results)} endpoints")
compliant = sum(1 for r in results if r.is_compliant)
print(f"Compliant: {compliant}/{len(results)}")

# Opción 3: Programmatic policy creation
from ai_gateway_policy_manager.models import Policy, PolicyRule

policy = Policy(
    name="disabled-endpoints",
    version="1.0",
    applies_to={"name": "^databricks-.*"},
    rules={
        "ai_gateway.rate_limits.user.requests_per_minute": PolicyRule(
            type="fixed",
            default=0,
            error_message="User rate limit must be 0"
        )
    }
)

result = manager.apply_policy(
    endpoint_name="my-endpoint",
    policy=policy,
    dry_mode=True
)
"""

print("✅ Este es el API objetivo - ver código comentado arriba")

In [0]:
# ============================================
# QUICK START: Crear estructura básica
# ============================================

import os

# Estructura de carpetas a crear
package_structure = """
ai_gateway_policy_manager/
├── pyproject.toml
├── README.md
├── src/
│   └── ai_gateway_policy_manager/
│       ├── __init__.py
│       ├── adapters/
│       │   ├── __init__.py
│       │   └── databricks_adapter.py
│       ├── core/
│       │   ├── __init__.py
│       │   ├── policy_engine.py
│       │   ├── policy_loader.py
│       │   └── filters.py
│       ├── models/
│       │   ├── __init__.py
│       │   ├── policy.py
│       │   └── results.py
│       └── manager.py
├── tests/
│   ├── __init__.py
│   └── test_policy_engine.py
└── examples/
    └── notebooks/
        └── example_usage.py
"""

print("📦 Estructura del paquete a crear:")
print(package_structure)

print("\n🔧 Comandos para empezar:")
print("""
# 1. Crear directorio del proyecto
mkdir -p ai_gateway_policy_manager
cd ai_gateway_policy_manager

# 2. Inicializar con Poetry (recomendado)
poetry init
poetry add databricks-sdk pandas
poetry add --group dev pytest black mypy

# O con pip/setuptools
python -m venv .venv
source .venv/bin/activate
pip install databricks-sdk pandas pytest

# 3. Crear estructura de carpetas
mkdir -p src/ai_gateway_policy_manager/{adapters,core,models}
mkdir -p tests examples/notebooks

# 4. Crear archivos __init__.py
touch src/ai_gateway_policy_manager/__init__.py
touch src/ai_gateway_policy_manager/adapters/__init__.py
touch src/ai_gateway_policy_manager/core/__init__.py
touch src/ai_gateway_policy_manager/models/__init__.py
""")

print("\n📝 Próximo paso: Copiar el código de este notebook a los módulos correspondientes")

## 📝 Ejemplo: `pyproject.toml`

```toml
[build-system]
requires = ["setuptools>=61.0", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "ai-gateway-policy-manager"
version = "0.1.0"
description = "Policy management for Databricks AI Gateway endpoints"
readme = "README.md"
requires-python = ">=3.9"
authors = [
    {name = "Your Team", email = "team@almirall.com"}
]
classifiers = [
    "Development Status :: 3 - Alpha",
    "Intended Audience :: Developers",
    "Programming Language :: Python :: 3.9",
    "Programming Language :: Python :: 3.10",
    "Programming Language :: Python :: 3.11",
]

dependencies = [
    "databricks-sdk>=0.20.0",
    "pandas>=1.5.0",
]

[project.optional-dependencies]
dev = [
    "pytest>=7.0",
    "pytest-cov>=4.0",
    "black>=23.0",
    "mypy>=1.0",
    "ruff>=0.1.0",
]

[project.scripts]
ai-gateway-policy = "ai_gateway_policy_manager.cli.main:main"

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
testpaths = ["tests"]
pythonpath = ["src"]

[tool.black]
line-length = 100
target-version = ['py39']

[tool.mypy]
python_version = "3.9"
strict = true
```

In [0]:
# ============================================
# EJEMPLO: Cómo quedaría policy_engine.py refactorizado
# ============================================

"""
Este es un ejemplo de cómo refactorizar el código actual
en un módulo profesional con type hints, docstrings, y error handling.
"""

from typing import Dict, Tuple, Any, List, Optional
from dataclasses import dataclass
import logging

logger = logging.getLogger(__name__)


@dataclass
class ValidationError:
    """Represents a policy validation error."""
    key: str
    message: str
    expected: Optional[Any] = None
    found: Optional[Any] = None


@dataclass
class PolicyResult:
    """Result of applying a policy to an endpoint configuration."""
    is_compliant: bool
    corrected_config: Dict[str, Any]
    errors: List[ValidationError]
    endpoint_name: Optional[str] = None


class PolicyEngine:
    """Engine for validating and applying policies to endpoint configurations."""

    @staticmethod
    def parse_rate_limit_key(policy_key: str) -> Dict[str, Any]:
        """
        Parse a policy key into components for rate limit manipulation.
        
        Args:
            policy_key: Policy key in dot notation (e.g., 'ai_gateway.rate_limits.user.requests_per_minute')
        
        Returns:
            Dictionary with parsed components:
                - is_rate_limit: Whether this is a rate limit policy
                - key_name: The rate limit key name (e.g., 'user')
                - field: The field name (e.g., 'requests_per_minute')
                - renewal_period: The renewal period (e.g., 'minute')
        
        Examples:
            >>> PolicyEngine.parse_rate_limit_key('ai_gateway.rate_limits.user.requests_per_minute')
            {'is_rate_limit': True, 'key_name': 'user', 'field': 'requests_per_minute', 'renewal_period': 'minute'}
        """
        parts = policy_key.split('.')
        
        if len(parts) >= 4 and parts[0] == 'ai_gateway' and parts[1] == 'rate_limits':
            key_name = parts[2]
            field = parts[3]
            renewal_period = field.replace('requests_per_', '') if field.startswith('requests_per_') else 'minute'
            
            return {
                'is_rate_limit': True,
                'key_name': key_name,
                'field': field,
                'renewal_period': renewal_period
            }
        
        return {'is_rate_limit': False}

    @staticmethod
    def search_nested_key(data: Dict[str, Any], key: str) -> Tuple[bool, Any]:
        """
        Search for a nested key in a dictionary using dot notation.
        
        Args:
            data: Dictionary to search
            key: Key in dot notation (e.g., 'config.rate_limit')
        
        Returns:
            Tuple of (exists, value) where exists is True if key found
        
        Examples:
            >>> PolicyEngine.search_nested_key({'config': {'rate_limit': 100}}, 'config.rate_limit')
            (True, 100)
        """
        keys = key.split(".")
        current = data
        
        for k in keys[:-1]:
            if isinstance(current, dict) and k in current:
                current = current[k]
            else:
                return False, None
        
        if isinstance(current, dict) and keys[-1] in current:
            return True, current[keys[-1]]
        
        return False, None

    def apply_policy(
        self,
        policy_rules: Dict[str, Dict[str, Any]],
        endpoint_config: Dict[str, Any]
    ) -> PolicyResult:
        """
        Apply policy rules to an endpoint configuration.
        
        Always applies corrections. Use dry_mode at a higher level to control
        whether corrected_config is actually applied to the endpoint.
        
        Args:
            policy_rules: Dictionary of policy rules
            endpoint_config: Endpoint configuration to validate
        
        Returns:
            PolicyResult with validation results and corrected configuration
        
        Raises:
            ValueError: If policy_rules or endpoint_config is invalid
        """
        if not isinstance(policy_rules, dict):
            raise ValueError("policy_rules must be a dictionary")
        if not isinstance(endpoint_config, dict):
            raise ValueError("endpoint_config must be a dictionary")
        
        errors: List[ValidationError] = []
        corrected_config = endpoint_config.copy()
        
        logger.info(f"Applying {len(policy_rules)} policy rules")
        
        for policy_key, rule in policy_rules.items():
            try:
                self._apply_single_rule(
                    policy_key=policy_key,
                    rule=rule,
                    corrected_config=corrected_config,
                    errors=errors
                )
            except Exception as e:
                logger.error(f"Error applying rule {policy_key}: {e}")
                errors.append(ValidationError(
                    key=policy_key,
                    message=f"Internal error: {str(e)}"
                ))
        
        is_compliant = len(errors) == 0
        logger.info(f"Policy application complete. Compliant: {is_compliant}, Errors: {len(errors)}")
        
        return PolicyResult(
            is_compliant=is_compliant,
            corrected_config=corrected_config,
            errors=errors
        )

    def _apply_single_rule(
        self,
        policy_key: str,
        rule: Dict[str, Any],
        corrected_config: Dict[str, Any],
        errors: List[ValidationError]
    ) -> None:
        """Apply a single policy rule (internal method)."""
        rule_type = rule.get("type")
        default = rule.get("default")
        error_message = rule.get("error_message", f"Policy violation for {policy_key}")
        
        parsed = self.parse_rate_limit_key(policy_key)
        
        if parsed['is_rate_limit']:
            self._apply_rate_limit_rule(
                policy_key=policy_key,
                rule_type=rule_type,
                default=default,
                error_message=error_message,
                parsed=parsed,
                corrected_config=corrected_config,
                errors=errors
            )
        else:
            self._apply_generic_rule(
                policy_key=policy_key,
                rule_type=rule_type,
                default=default,
                error_message=error_message,
                corrected_config=corrected_config,
                errors=errors
            )

    # ... (resto de métodos privados)


print("✅ Ejemplo de código refactorizado - ver arriba")
print("\nMejoras:")
print("  ✓ Type hints completos")
print("  ✓ Docstrings con ejemplos")
print("  ✓ Dataclasses para resultados")
print("  ✓ Logging estructurado")
print("  ✓ Error handling robusto")
print("  ✓ Métodos privados separados")

## 📦 Opciones de Distribución

### Opción 1: Repositorio Privado (Recomendado para Empresa)

**Azure Artifacts** (si usáis Azure DevOps):
```bash
# Build
poetry build
# o
python -m build

# Publicar a Azure Artifacts
twine upload --repository-url https://pkgs.dev.azure.com/almirall/_packaging/my-feed/pypi/upload/ dist/*

# Instalar desde notebook
%pip install ai-gateway-policy-manager --index-url https://pkgs.dev.azure.com/almirall/_packaging/my-feed/pypi/simple/
```

### Opción 2: DBFS/Volumes (Más simple para Databricks)

```bash
# Build wheel
poetry build

# Subir a Volumes
dbfs cp dist/ai_gateway_policy_manager-0.1.0-py3-none-any.whl /Volumes/main/shared/packages/

# Instalar desde notebook
%pip install /Volumes/main/shared/packages/ai_gateway_policy_manager-0.1.0-py3-none-any.whl
```

### Opción 3: Git + pip (Para desarrollo)

```bash
# Instalar directamente desde Git
%pip install git+https://github.com/almirall/ai-gateway-policy-manager.git@main

# O desde una rama específica
%pip install git+https://github.com/almirall/ai-gateway-policy-manager.git@feature/new-rules
```

### Opción 4: Workspace Files (Más simple, sin build)

```python
# Añadir al sys.path en el notebook
import sys
sys.path.append('/Workspace/Users/tromerorodri@almirall.com/ai_gateway_policy_manager/src')

from ai_gateway_policy_manager import PolicyManager
```

**Recomendación**: Empezar con Opción 2 (DBFS/Volumes) para prototipo, luego migrar a Opción 1 (Azure Artifacts) para producción.

In [0]:
# ============================================
# RESUMEN: Próximos pasos inmediatos
# ============================================

print("🎯 NEXT ACTIONS - Empezar HOY:\n")

steps = [
    {
        "priority": "🔴 ALTA",
        "task": "1. Crear estructura del paquete",
        "time": "30 min",
        "actions": [
            "mkdir ai_gateway_policy_manager",
            "Crear carpetas: src/, tests/, examples/",
            "Crear pyproject.toml básico"
        ]
    },
    {
        "priority": "🔴 ALTA",
        "task": "2. Refactorizar código core",
        "time": "2-3 horas",
        "actions": [
            "Copiar apply_policy_v4 → core/policy_engine.py",
            "Copiar DatabricksEndpointAdapter → adapters/databricks_adapter.py",
            "Añadir type hints y docstrings",
            "Crear dataclasses para PolicyResult, ValidationError"
        ]
    },
    {
        "priority": "🟡 MEDIA",
        "task": "3. Crear PolicyManager (API principal)",
        "time": "2 horas",
        "actions": [
            "Crear manager.py con clase PolicyManager",
            "Implementar load_policy(), apply_policy(), apply_policy_bulk()",
            "Añadir parámetro dry_mode"
        ]
    },
    {
        "priority": "🟡 MEDIA",
        "task": "4. Build & Test local",
        "time": "1 hora",
        "actions": [
            "poetry build o python -m build",
            "pip install dist/*.whl en entorno limpio",
            "Probar import y uso básico"
        ]
    },
    {
        "priority": "🟢 BAJA",
        "task": "5. Distribuir a Volumes/DBFS",
        "time": "30 min",
        "actions": [
            "Subir wheel a /Volumes/main/shared/packages/",
            "Crear notebook de ejemplo",
            "Documentar instalación"
        ]
    }
]

for step in steps:
    print(f"{step['priority']} {step['task']} ({step['time']})")
    for action in step['actions']:
        print(f"    • {action}")
    print()

print("💡 RECOMENDACIÓN:")
print("   Empezar con pasos 1-2 esta semana")
print("   Objetivo: Tener un wheel funcional instalable en 3-5 días")
print("\n📝 DOCUMENTACIÓN:")
print("   Crear README.md con:")
print("     - Instalación")
print("     - Ejemplos de uso")
print("     - Estructura de políticas")
print("     - API reference")

## 🔧 Setup con Pipenv

### Estructura del Paquete (con Pipenv)

```
ai_gateway_policy_manager/
├── Pipfile                      # Dependencias (reemplaza pyproject.toml para deps)
├── Pipfile.lock                 # Lock file generado automáticamente
├── setup.py                     # Configuración del paquete para build
├── README.md
├── src/
│   └── ai_gateway_policy_manager/
│       ├── __init__.py
│       ├── adapters/
│       │   ├── __init__.py
│       │   └── databricks_adapter.py
│       ├── core/
│       │   ├── __init__.py
│       │   ├── policy_engine.py
│       │   ├── policy_loader.py
│       │   └── filters.py
│       ├── models/
│       │   ├── __init__.py
│       │   ├── policy.py
│       │   └── results.py
│       └── manager.py
├── tests/
│   ├── __init__.py
│   └── test_policy_engine.py
└── examples/
    └── notebooks/
        └── example_usage.py
```

### Comandos Iniciales

```bash
# 1. Crear directorio del proyecto
mkdir ai_gateway_policy_manager
cd ai_gateway_policy_manager

# 2. Inicializar pipenv
pipenv --python 3.10

# 3. Instalar dependencias de producción
pipenv install databricks-sdk pandas

# 4. Instalar dependencias de desarrollo
pipenv install --dev pytest pytest-cov black mypy ruff

# 5. Crear estructura de carpetas
mkdir -p src/ai_gateway_policy_manager/{adapters,core,models}
mkdir -p tests examples/notebooks

# 6. Crear archivos __init__.py
touch src/ai_gateway_policy_manager/__init__.py
touch src/ai_gateway_policy_manager/adapters/__init__.py
touch src/ai_gateway_policy_manager/core/__init__.py
touch src/ai_gateway_policy_manager/models/__init__.py
touch tests/__init__.py
```

## 📝 Ejemplo: `Pipfile`

```toml
[[source]]
url = "https://pypi.org/simple"
verify_ssl = true
name = "pypi"

[packages]
databricks-sdk = ">=0.20.0"
pandas = ">=1.5.0"

[dev-packages]
pytest = ">=7.0"
pytest-cov = ">=4.0"
black = ">=23.0"
mypy = ">=1.0"
ruff = ">=0.1.0"
build = "*"  # Para crear el wheel
twine = "*"  # Para subir a repositorios

[requires]
python_version = "3.10"

[scripts]
test = "pytest tests/ -v"
test-cov = "pytest tests/ --cov=src/ai_gateway_policy_manager --cov-report=html"
lint = "ruff check src/ tests/"
format = "black src/ tests/"
type-check = "mypy src/"
build = "python -m build"
```

### Uso de scripts definidos:

```bash
# Ejecutar tests
pipenv run test

# Tests con coverage
pipenv run test-cov

# Formatear código
pipenv run format

# Linting
pipenv run lint

# Type checking
pipenv run type-check

# Build wheel
pipenv run build
```

## 📝 Ejemplo: `setup.py`

Con pipenv, necesitas un `setup.py` para crear el wheel:

```python
from setuptools import setup, find_packages

with open("README.md", "r", encoding="utf-8") as fh:
    long_description = fh.read()

setup(
    name="ai-gateway-policy-manager",
    version="0.1.0",
    author="Your Team",
    author_email="team@almirall.com",
    description="Policy management for Databricks AI Gateway endpoints",
    long_description=long_description,
    long_description_content_type="text/markdown",
    url="https://github.com/almirall/ai-gateway-policy-manager",
    package_dir={"":"src"},
    packages=find_packages(where="src"),
    classifiers=[
        "Development Status :: 3 - Alpha",
        "Intended Audience :: Developers",
        "Programming Language :: Python :: 3.9",
        "Programming Language :: Python :: 3.10",
        "Programming Language :: Python :: 3.11",
    ],
    python_requires=">=3.9",
    install_requires=[
        "databricks-sdk>=0.20.0",
        "pandas>=1.5.0",
    ],
    extras_require={
        "dev": [
            "pytest>=7.0",
            "pytest-cov>=4.0",
            "black>=23.0",
            "mypy>=1.0",
            "ruff>=0.1.0",
        ],
    },
    entry_points={
        "console_scripts": [
            "ai-gateway-policy=ai_gateway_policy_manager.cli.main:main",
        ],
    },
)
```

## 🔄 Workflow de Desarrollo con Pipenv

### 1. Setup Inicial

```bash
# Clonar/crear proyecto
git clone <repo> ai_gateway_policy_manager
cd ai_gateway_policy_manager

# Instalar dependencias (crea virtualenv automáticamente)
pipenv install --dev

# Activar el virtualenv
pipenv shell
```

### 2. Desarrollo

```bash
# Trabajar dentro del virtualenv
pipenv shell

# O ejecutar comandos sin activar
pipenv run pytest
pipenv run black src/
```

### 3. Añadir Dependencias

```bash
# Dependencia de producción
pipenv install nueva-libreria

# Dependencia de desarrollo
pipenv install --dev nueva-dev-tool

# Desde requirements.txt (si migras código existente)
pipenv install -r requirements.txt
```

### 4. Build & Distribution

```bash
# Instalar herramientas de build (si no están)
pipenv install --dev build twine

# Crear wheel
pipenv run python -m build
# Genera: dist/ai_gateway_policy_manager-0.1.0-py3-none-any.whl

# Verificar el wheel
pipenv run twine check dist/*
```

### 5. Testing

```bash
# Tests básicos
pipenv run pytest

# Con coverage
pipenv run pytest --cov=src/ai_gateway_policy_manager --cov-report=html

# Ver reporte
open htmlcov/index.html
```

### 6. Instalación Local para Testing

```bash
# Salir del virtualenv del proyecto
exit

# Crear un nuevo virtualenv limpio para testing
mkdir test_install && cd test_install
pipenv --python 3.10

# Instalar el wheel
pipenv install ../ai_gateway_policy_manager/dist/ai_gateway_policy_manager-0.1.0-py3-none-any.whl

# Probar import
pipenv run python -c "from ai_gateway_policy_manager import PolicyManager; print('✅ Import OK')"
```

In [0]:
# ============================================
# NEXT ACTIONS ACTUALIZADOS - Con Pipenv
# ============================================

print("🎯 NEXT ACTIONS - Empezar HOY (con Pipenv):\n")

steps = [
    {
        "priority": "🔴 ALTA",
        "task": "1. Crear estructura del paquete con Pipenv",
        "time": "30 min",
        "actions": [
            "mkdir ai_gateway_policy_manager && cd ai_gateway_policy_manager",
            "pipenv --python 3.10",
            "pipenv install databricks-sdk pandas",
            "pipenv install --dev pytest black mypy build",
            "Crear carpetas: src/, tests/, examples/",
            "Crear setup.py básico"
        ]
    },
    {
        "priority": "🔴 ALTA",
        "task": "2. Refactorizar código core",
        "time": "2-3 horas",
        "actions": [
            "Copiar apply_policy_v4 → src/ai_gateway_policy_manager/core/policy_engine.py",
            "Copiar DatabricksEndpointAdapter → src/ai_gateway_policy_manager/adapters/databricks_adapter.py",
            "Copiar read_policy, apply_to_filter → módulos correspondientes",
            "Añadir type hints y docstrings",
            "Crear dataclasses para PolicyResult, ValidationError"
        ]
    },
    {
        "priority": "🟡 MEDIA",
        "task": "3. Crear PolicyManager (API principal)",
        "time": "2 horas",
        "actions": [
            "Crear src/ai_gateway_policy_manager/manager.py",
            "Implementar load_policy(), apply_policy(), apply_policy_bulk()",
            "Añadir parámetro dry_mode",
            "Añadir logging"
        ]
    },
    {
        "priority": "🟡 MEDIA",
        "task": "4. Build & Test con Pipenv",
        "time": "1 hora",
        "actions": [
            "pipenv run python -m build",
            "Crear virtualenv limpio para testing",
            "pipenv install dist/*.whl en entorno limpio",
            "Probar import y uso básico"
        ]
    },
    {
        "priority": "🟢 BAJA",
        "task": "5. Distribuir a Volumes/DBFS",
        "time": "30 min",
        "actions": [
            "Subir wheel a /Volumes/main/shared/packages/",
            "Crear notebook de ejemplo en examples/notebooks/",
            "Documentar instalación en README.md"
        ]
    }
]

for step in steps:
    print(f"{step['priority']} {step['task']} ({step['time']})")
    for action in step['actions']:
        print(f"    • {action}")
    print()

print("💡 VENTAJAS DE PIPENV:")
print("   ✓ Gestión automática de virtualenv")
print("   ✓ Pipfile.lock asegura reproducibilidad")
print("   ✓ Scripts personalizados en Pipfile")
print("   ✓ Separación clara dev/prod dependencies")
print("   ✓ Compatible con requirements.txt (migración fácil)")

print("\n📋 COMANDOS RÁPIDOS:")
print("   pipenv install          # Instalar deps")
print("   pipenv shell            # Activar virtualenv")
print("   pipenv run pytest       # Ejecutar tests")
print("   pipenv run build        # Crear wheel")
print("   pipenv graph            # Ver árbol de dependencias")